# Interactive Table Tennis Simulation

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Video
import benchmark_direct_services as benchmark_direct
import table_tennis_simulation as tts

benchmark_cases = list(benchmark_direct.build_cases())
services = sorted({case.service for case in benchmark_cases})
depths = list(benchmark_direct.DEPTHS.keys())
lanes = list(benchmark_direct.LANES.keys())

service_dropdown = widgets.Dropdown(options=services, value='pendulum', description='Servicio')
depth_dropdown = widgets.Dropdown(options=depths, value='short', description='Profundidad')
lane_dropdown = widgets.Dropdown(options=lanes, value='forehand', description='Zona')

pos_x = widgets.FloatSlider(value=0, min=-500, max=500, step=10, description='pos_x')
pos_y = widgets.FloatSlider(value=tts.TABLE_WIDTH*4/8, min=0, max=tts.TABLE_WIDTH, step=10, description='pos_y')
pos_z = widgets.FloatSlider(value=tts.TABLE_HEIGHT + 2*tts.NET_HEIGHT, min=tts.TABLE_HEIGHT, max=tts.TABLE_HEIGHT+400, step=10, description='pos_z')
vel_x = widgets.FloatSlider(value=7000, min=-10000, max=10000, step=100, description='vel_x')
vel_y = widgets.FloatSlider(value=-3000, min=-10000, max=10000, step=100, description='vel_y')
vel_z = widgets.FloatSlider(value=-3000, min=-10000, max=10000, step=100, description='vel_z')
omega_x = widgets.FloatSlider(value=0, min=-100, max=100, step=1, description='spin_x')
omega_y = widgets.FloatSlider(value=0, min=-100, max=100, step=1, description='spin_y')
omega_z = widgets.FloatSlider(value=75, min=-100, max=100, step=1, description='spin_z')

run_btn = widgets.Button(description='Run')
save_btn = widgets.Button(description='Save MP4')
out = widgets.Output()


def selected_benchmark_case():
    for case in benchmark_cases:
        if (case.service, case.depth, case.lane) == (service_dropdown.value, depth_dropdown.value, lane_dropdown.value):
            return case
    raise ValueError('No existe esa combinación en el benchmark')


def apply_benchmark_preset(_=None):
    ic = selected_benchmark_case().initial_conditions
    pos_x.value, pos_y.value, pos_z.value = ic.pos
    vel_x.value, vel_y.value, vel_z.value = ic.vel
    omega_x.value, omega_y.value, omega_z.value = tuple(tts.np.array(ic.omega) / (2 * tts.np.pi))


for dropdown in (service_dropdown, depth_dropdown, lane_dropdown):
    dropdown.observe(apply_benchmark_preset, names='value')


def build_ic():
    return tts.InitialConditions(
        pos=(pos_x.value, pos_y.value, pos_z.value),
        vel=(vel_x.value, vel_y.value, vel_z.value),
        omega=(omega_x.value*2*tts.np.pi, omega_y.value*2*tts.np.pi, omega_z.value*2*tts.np.pi),
    )


def on_run(_):
    ic = build_ic()
    result = tts.simulate(ic)
    with out:
        clear_output(wait=True)
        tts.animate_simulation(result)


def on_save(_):
    ic = build_ic()
    result = tts.simulate(ic)
    tts.animate_simulation(result, save='simulation.mp4')
    with out:
        clear_output(wait=True)
        display(Video('simulation.mp4'))


run_btn.on_click(on_run)
save_btn.on_click(on_save)
apply_benchmark_preset()

ui = widgets.VBox([
    widgets.HBox([service_dropdown, depth_dropdown, lane_dropdown]),
    pos_x, pos_y, pos_z,
    vel_x, vel_y, vel_z,
    omega_x, omega_y, omega_z,
    widgets.HBox([run_btn, save_btn]),
    out
])

display(ui)
